## Imports & Setup

In [1]:
import os, warnings
warnings.filterwarnings('ignore')

CFG = dict(
    SEED          = 42,
    N_FOLDS       = 2,

    EPOCHS_ELECTRA     = 2,
    BS_ELECTRA         = 4,
    GRAD_ACCUM_ELECTRA = 4,        
    MAX_LEN_ELECTRA    = 384,      
    LR_ELECTRA         = 8e-6,    
    WARMUP_RATIO       = 0.1,     
 
    MARGIN        = 0.5,
    USE_BF16      = True,          
    
    DATA_DIR      = '/kaggle/input/competitions/smart-mcq-solver-challenge',
    OUTPUT_DIR    = '/kaggle/working',
    ARTIFACT_DIR  = '/kaggle/working/artifacts',
    ELECTRA_MODEL = 'google/electra-large-discriminator'
)

os.makedirs(CFG['OUTPUT_DIR'], exist_ok=True)
os.makedirs(CFG['ARTIFACT_DIR'], exist_ok=True)
OPTS = ['A', 'B', 'C', 'D', 'E']


In [2]:
import os, sys, json, math, random, time, gc, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast as _autocast, GradScaler as _GradScaler

from sklearn.metrics import f1_score, accuracy_score


def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(CFG['SEED'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

AMP_DTYPE = torch.float16
USE_SCALER = True
print(f"AMP dtype: {AMP_DTYPE} | GradScaler: {USE_SCALER}")

def autocast():
    return _autocast(device_type='cuda', dtype=AMP_DTYPE)

def make_scaler():
    return _GradScaler('cuda')

Device: cuda
AMP dtype: torch.float16 | GradScaler: True


## Load Data

In [3]:
train = pd.read_csv(f"{CFG['DATA_DIR']}/train.csv")
test  = pd.read_csv(f"{CFG['DATA_DIR']}/test.csv")
print(f"train: {train.shape}, test: {test.shape}")



train: (2000, 8), test: (500, 7)


In [4]:
y_idx = np.array([OPTS.index(a) for a in train['answer']])

## Normalize + GroupKFold Assignment

In [5]:
from sklearn.model_selection import GroupKFold

_BOILERPLATE_PAT = re.compile(
    r"pick the best possible answer:?|\bcarefully\b\.?|among the listed options\.?|"
    r"choose the (correct|best) (option|answer|response)\.?|"
    r"select the (best|correct|most appropriate) (option|answer|response)\.?|"
    r"determine the correct option:?|identify the correct statement:?|"
    r"based on the given context\.?|which of the following( statements)?( is true)?",
    flags=re.IGNORECASE
)
_WS = re.compile(r'\s+')

def normalize_text(text):
    s = _BOILERPLATE_PAT.sub(' ', str(text))
    return _WS.sub(' ', s).strip()

# Apply normalization
for df in (train, test):
    df['prompt_n'] = df['prompt'].map(normalize_text)
    for o in OPTS:
        df[f'{o}_n'] = df[o].map(normalize_text)

# GroupKFold assignment
gkf = GroupKFold(n_splits=CFG['N_FOLDS'])
train['fold'] = -1
for fold_id, (_, va_idx) in enumerate(gkf.split(train, groups=train['prompt_n'])):
    train.iloc[va_idx, train.columns.get_loc('fold')] = fold_id

# Leakage Checks
assert (train['fold'] >= 0).all() and train.groupby('prompt_n')['fold'].nunique().max() == 1, "Leak detected!"
print("Folds:\n", train['fold'].value_counts().sort_index().to_string())

Folds:
 fold
0    1000
1    1000


## Metrics & Loss 

In [15]:

def map_at_3(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx     = np.asarray(y_idx)
    order = (-scores_2d).argsort(axis=1)
    total = 0.0
    for i in range(len(scores_2d)):
        top3 = order[i, :3]
        if y_idx[i] in top3:
            total += 1.0 / (list(top3).index(y_idx[i]) + 1)
    return total / len(scores_2d)

def all_metrics(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx     = np.asarray(y_idx)
    pred_top1 = scores_2d.argmax(axis=1)
    return dict(
        map3 = map_at_3(scores_2d, y_idx),
        acc  = accuracy_score(y_idx, pred_top1),
        f1   = f1_score(y_idx, pred_top1, average='macro'),
    )


def pairwise_margin_loss(scores, correct, margin=0.5):
    B = scores.size(0)
    pos = scores[torch.arange(B), correct].unsqueeze(1).expand(-1, 5)
    target = torch.ones_like(scores)
    loss = F.margin_ranking_loss(pos, scores, target, margin=margin, reduction='none')
    mask = torch.ones_like(loss); mask[torch.arange(B), correct] = 0
    return (loss * mask).sum() / mask.sum().clamp(min=1)


print("Metrics and loss defined.")

Metrics and loss defined.


## W&B Setup

In [16]:
import os
try:
    import wandb
except ImportError:
    wandb = None


USE_WANDB     = True
RUN_VERSION   = 5
RUN_NOTES     = "electra_run_5, n_folds=2, epoch=2"
WANDB_PROJECT = "24f1000134-t22026"

if USE_WANDB and wandb is not None:
    try:
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret("wandb_api"))
        print("W&B connected.")
    except Exception as e:
        print(f"W&B login skipped ({e}); set Kaggle secret 'wandb_api' to enable.")
        USE_WANDB = False
elif wandb is None:
    USE_WANDB = False
    print("wandb not installed; logging disabled.")

def wandb_init(model_tag, fold):
    if not (USE_WANDB and wandb is not None):
        return None
    return wandb.init(
        project=WANDB_PROJECT,
        name=f"{model_tag}_v{RUN_VERSION}_fold{fold}",
        group=f"{model_tag}_v{RUN_VERSION}",          
        job_type="train",
        notes=RUN_NOTES,                           
        tags=[model_tag, f"v{RUN_VERSION}"],
        config={**CFG, "run_version": RUN_VERSION,
                "run_notes": RUN_NOTES, "model": model_tag, "fold": fold},
        reinit=True,
    )

def wlog(metrics, step=None):
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.log(metrics, step=step)

def wandb_finish():
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f1000134 (24f1000134-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B connected.


### Uploading Model to Kagglehub

In [17]:
KAGGLE_USERNAME = "kamal134134"
PUSH_TO_HUB     = True
HUB_FRAMEWORK   = "pyTorch"

HUB_HANDLES = {
    "electra": f"{KAGGLE_USERNAME}/genai_project/{HUB_FRAMEWORK}/electra"
}

def push_model(model_tag, folder, notes=""):
    if not PUSH_TO_HUB or kagglehub is None:
        print(f"[hub] upload skipped for '{model_tag}' (PUSH_TO_HUB=False).")
        return
    handle = HUB_HANDLES[model_tag]
    note = notes or f"v{RUN_VERSION}: {RUN_NOTES}"
    print(f"[hub] uploading '{model_tag}' -> {handle}")
    kagglehub.model_upload(handle, folder, version_notes=note)
    print("Done. Inference notebooks can now kagglehub.")

## HF Auth


In [18]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF token loaded.")
else:
    print("No HF token found — public models will still download, gated ones won't.")

HF token loaded.


In [21]:
print("Downloading electra-large-discriminator tokenizer...")
t = time.time()
_tok = AutoTokenizer.from_pretrained(CFG['ELECTRA_MODEL'], token=os.environ['HF_TOKEN'])
print(f"  tokenizer ({time.time()-t:.0f}s)")

print("Downloading electra-large-discriminator model weights (~1.3 GB)...")
t = time.time()
_mdl = AutoModel.from_pretrained(CFG['ELECTRA_MODEL'], token=os.environ['HF_TOKEN'])
print(f"  model ({time.time()-t:.0f}s)")

del _mdl, _tok; gc.collect(); torch.cuda.empty_cache()
print("Models cached.")


config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  tokenizer (2s)


pytorch_model.bin:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: google/electra-large-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
electra.embeddings_project.bias                   | UNEXPECTED |  | 
electra.embeddings_project.weight                 | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  model (18s)
Models cached.


## Model and Dataset 

In [22]:
class ElectraScorer(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.backbone.gradient_checkpointing_enable() 
        h = self.backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(h, h // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(h // 2, 1),
        )

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls).squeeze(-1)


class ElectraDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, context_map=None,
                 shuffle_opts=False, has_label=True, seed=0):
        self.df       = df.reset_index(drop=True)
        self.tok      = tokenizer
        self.max_len  = max_len
        self.ctx      = context_map or {}
        self.shuffle  = shuffle_opts
        self.has_label = has_label
        self.rng      = np.random.default_rng(seed)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        prompt = r['prompt_n']
        if r['id'] in self.ctx:
            prompt = self.ctx[r['id']] + ' ' + prompt
        options = [r[f'{o}_n'] for o in OPTS]
        correct = OPTS.index(r['answer']) if self.has_label else -1
        if self.shuffle:
            perm = self.rng.permutation(5)
            options = [options[i] for i in perm]
            if self.has_label:
                correct = int(np.where(perm == correct)[0][0])
        enc = self.tok([prompt]*5, options, truncation=True,
                       padding='max_length', max_length=self.max_len,
                       return_tensors='pt')
        return dict(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            correct=torch.tensor(correct, dtype=torch.long),
            id=int(r['id']),
        )


## Training Function

In [28]:
def train_electra_cv(train, test, n_folds=None):
    if n_folds is None: n_folds = CFG['N_FOLDS']

    MODEL_TAG = 'electra'

    artifact_dir = f"{CFG['ARTIFACT_DIR']}/{MODEL_TAG}"
    os.makedirs(artifact_dir, exist_ok=True)

    oof_scores  = np.zeros((len(train), 5), dtype=np.float32)
    test_scores = np.zeros((len(test),  5), dtype=np.float32)
    y_idx       = np.array([OPTS.index(a) for a in train['answer']])

    tokenizer = AutoTokenizer.from_pretrained(CFG['ELECTRA_MODEL'])

    assert 'fold' in train.columns, "train must have a 'fold' column"

    for fold in range(n_folds):
  
        run = wandb_init(MODEL_TAG, fold)

        va_idx = np.where(train['fold'].values == fold)[0]
        tr_idx = np.where(train['fold'].values != fold)[0]
        print(f'\n[ELECTRA] === fold {fold} (train={len(tr_idx)}, val={len(va_idx)}) ===', flush=True)

        tr_ds = ElectraDataset(train.iloc[tr_idx], tokenizer, CFG['MAX_LEN_ELECTRA'],
                               CONTEXT_MAP, shuffle_opts=True,  seed=fold)
        va_ds = ElectraDataset(train.iloc[va_idx], tokenizer, CFG['MAX_LEN_ELECTRA'],
                               CONTEXT_MAP, shuffle_opts=False)
        te_ds = ElectraDataset(test,              tokenizer, CFG['MAX_LEN_ELECTRA'],
                               CONTEXT_MAP, shuffle_opts=False, has_label=False)

        tr_dl = DataLoader(tr_ds, batch_size=CFG['BS_ELECTRA'], shuffle=True,
                           num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, batch_size=CFG['BS_ELECTRA'], shuffle=False,
                           num_workers=0, pin_memory=True)
        te_dl = DataLoader(te_ds, batch_size=CFG['BS_ELECTRA'], shuffle=False,
                           num_workers=0, pin_memory=True)

        model  = ElectraScorer(CFG['ELECTRA_MODEL']).float().to(DEVICE)
        optim  = AdamW(model.parameters(), lr=CFG['LR_ELECTRA'], weight_decay=0.01)
        steps  = CFG['EPOCHS_ELECTRA'] * len(tr_dl) // CFG['GRAD_ACCUM_ELECTRA']
        warmup = int(CFG['WARMUP_RATIO'] * steps)
        
        sched  = get_cosine_schedule_with_warmup(optim, num_warmup_steps=warmup,
                                                 num_training_steps=max(steps, 1))
        
        scaler = make_scaler()
        accum  = CFG['GRAD_ACCUM_ELECTRA']

        best_val_map3   = -1
        best_va_scores  = None
        best_te_scores  = None

        for epoch in range(CFG['EPOCHS_ELECTRA']):
            t_epoch    = time.time()
            model.train()
            train_loss = 0.0; n_train = 0
            optim.zero_grad()

            for step, batch in enumerate(tr_dl):
                B    = batch['input_ids'].size(0)
                ids  = batch['input_ids'].view(B*5, -1).to(DEVICE, non_blocking=True)
                mask = batch['attention_mask'].view(B*5, -1).to(DEVICE, non_blocking=True)
                correct = batch['correct'].to(DEVICE, non_blocking=True)

             
                with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE,
                                        enabled=(DEVICE.type == 'cuda')):
                    scores = model(ids, mask).view(B, 5)
                    loss   = pairwise_margin_loss(scores, correct, margin=CFG['MARGIN']) / accum

                if torch.isfinite(loss):
                    if USE_SCALER:
                        scaler.scale(loss).backward()
                    else:
                        loss.backward()
                else:
                    print(f'    [warn] step {step}: non-finite loss, skipping backward')

                if (step + 1) % accum == 0:
                    if USE_SCALER:
                        scaler.unscale_(optim)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    if USE_SCALER:
                        scaler.step(optim); scaler.update()
                    else:
                        optim.step()
                    sched.step()
                    optim.zero_grad()

                train_loss += loss.item() * B * accum; n_train += B

                if step % 25 == 0:
                    print(f'    step {step}/{len(tr_dl)}  '
                          f'loss={loss.item()*accum:.4f}  '
                          f'elapsed={time.time()-t_epoch:.0f}s', flush=True)

            model.eval()
            va_loss = 0.0; n_val = 0
            va_scores_all = []
            with torch.no_grad():
                for batch in va_dl:
                    B    = batch['input_ids'].size(0)
                    ids  = batch['input_ids'].view(B*5, -1).to(DEVICE)
                    mask = batch['attention_mask'].view(B*5, -1).to(DEVICE)
                    correct = batch['correct'].to(DEVICE)
                    with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE,
                                            enabled=(DEVICE.type == 'cuda')):
                        scores = model(ids, mask).view(B, 5)
                        loss   = pairwise_margin_loss(scores, correct, margin=CFG['MARGIN'])
                    va_loss += loss.item() * B; n_val += B
                    va_scores_all.append(scores.cpu().float().numpy())
            va_scores_all = np.concatenate(va_scores_all)
            val_m = all_metrics(va_scores_all, y_idx[va_idx])

            tr_loss_val = train_loss / max(n_train, 1)
            va_loss_val = va_loss    / max(n_val,   1)

            print(f'  ep{epoch} ({time.time()-t_epoch:.0f}s)  '
                  f'train_loss={tr_loss_val:.4f} '
                  f'val_loss={va_loss_val:.4f} val_map3={val_m["map3"]:.4f} '
                  f'val_acc={val_m["acc"]:.4f} val_f1={val_m["f1"]:.4f}', flush=True)

            wlog({
                'ele/train_loss': tr_loss_val,
                'ele/val_loss':   va_loss_val,
                'ele/val_map3':   val_m['map3'],
                'ele/val_acc':    val_m['acc'],
                'ele/val_f1':     val_m['f1'],
                'ele/fold':       fold,
                'ele/epoch':      epoch,
            })

            if val_m['map3'] > best_val_map3:
                best_val_map3  = val_m['map3']
                best_va_scores = va_scores_all

                model.eval()
                te_scores = []
                with torch.no_grad():
                    for batch in te_dl:
                        B    = batch['input_ids'].size(0)
                        ids  = batch['input_ids'].view(B*5, -1).to(DEVICE)
                        mask = batch['attention_mask'].view(B*5, -1).to(DEVICE)
                        with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE,
                                                enabled=(DEVICE.type == 'cuda')):
                            scores = model(ids, mask).view(B, 5)
                        te_scores.append(scores.cpu().float().numpy())
                best_te_scores = np.concatenate(te_scores)

                torch.save(model.state_dict(), f"{artifact_dir}/best_electra_fold{fold}.pth")
                print(f"    -> new best (val_map3={best_val_map3:.4f}), checkpoint saved", flush=True)

        wlog({'ele/best_val_map3': best_val_map3})
        oof_scores[va_idx] = best_va_scores
        test_scores        += best_te_scores / n_folds
        wandb_finish()

        del model; gc.collect(); torch.cuda.empty_cache()

    overall = all_metrics(oof_scores, y_idx)
    print(f'\n[ELECTRA] OOF MAP@3={overall["map3"]:.4f} '
          f'acc={overall["acc"]:.4f} f1={overall["f1"]:.4f}', flush=True)


    if USE_WANDB and wandb is not None:
        summary_run = wandb.init(
            project=WANDB_PROJECT,
            name=f"{MODEL_TAG}_v{RUN_VERSION}_OOF",
            group=f"{MODEL_TAG}_v{RUN_VERSION}",
            job_type="summary", tags=[MODEL_TAG, f"v{RUN_VERSION}", "oof"],
            reinit=True,
        )
        wandb.log({
            'ele/oof_map3': overall['map3'],
            'ele/oof_acc':  overall['acc'],
            'ele/oof_f1':   overall['f1'],
        })
        wandb.run.summary['oof_map3'] = overall['map3']
        wandb.run.summary['oof_acc']  = overall['acc']
        wandb.run.summary['oof_f1']   = overall['f1']
        wandb.finish()

 
    np.save(f"{artifact_dir}/oof_ele.npy",  oof_scores)
    np.save(f"{artifact_dir}/test_ele.npy", test_scores)
    print(f"Saved oof_ele.npy + test_ele.npy in {artifact_dir}")

    push_model(
        MODEL_TAG,
        artifact_dir,
        notes=f"v{RUN_VERSION}: {RUN_NOTES} | OOF MAP@3={overall['map3']:.4f} "
              f"acc={overall['acc']:.4f} f1={overall['f1']:.4f}"
    )

    return oof_scores, test_scores

oof_ele, test_ele = train_electra_cv(train, test, n_folds=CFG['N_FOLDS'])
print(f"\nTraining complete! oof_ele shape: {oof_ele.shape}, test_ele shape: {test_ele.shape}")


[ELECTRA] === fold 0 (train=1000, val=1000) ===


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: google/electra-large-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
electra.embeddings_project.bias                   | UNEXPECTED |  | 
electra.embeddings_project.weight                 | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


    step 0/250  loss=0.5321  elapsed=10s
    step 25/250  loss=0.4685  elapsed=248s
    step 50/250  loss=0.4666  elapsed=483s
    step 75/250  loss=0.4809  elapsed=719s
    step 100/250  loss=0.4460  elapsed=955s
    step 125/250  loss=0.4904  elapsed=1191s
    step 150/250  loss=0.4580  elapsed=1428s
    step 175/250  loss=0.3318  elapsed=1663s
    step 200/250  loss=0.3836  elapsed=1899s
    step 225/250  loss=0.3516  elapsed=2135s
  ep0 (3022s)  train_loss=0.4434 val_loss=0.3185 val_map3=0.8247 val_acc=0.7160 val_f1=0.7138
    -> new best (val_map3=0.8247), checkpoint saved
    step 0/250  loss=0.2970  elapsed=9s
    step 25/250  loss=0.2702  elapsed=245s
    step 50/250  loss=0.2109  elapsed=481s
    step 75/250  loss=0.2600  elapsed=716s
    step 100/250  loss=0.1418  elapsed=951s
    step 125/250  loss=0.1324  elapsed=1186s
    step 150/250  loss=0.0571  elapsed=1421s
    step 175/250  loss=0.3217  elapsed=1656s
    step 200/250  loss=0.1212  elapsed=1891s
    step 225/250  loss

ele/best_val_map3,▁
ele/epoch,▁█
ele/fold,▁▁
ele/train_loss,█▁
ele/val_acc,▁█
ele/val_f1,▁█
ele/val_loss,█▁
ele/val_map3,▁█
ele/best_val_map3,0.87283
ele/epoch,1
ele/fold,0


...

ele/best_val_map3,▁
ele/epoch,▁█
ele/fold,▁▁
ele/train_loss,█▁
ele/val_acc,▁█
ele/val_f1,▁█
ele/val_loss,█▁
ele/val_map3,▁█
ele/best_val_map3,0.875
ele/epoch,1
ele/fold,1



[ELECTRA] OOF MAP@3=0.8739 acc=0.7925 f1=0.7895


ele/oof_acc,▁
ele/oof_f1,▁
ele/oof_map3,▁
ele/oof_acc,0.7925
ele/oof_f1,0.78947
ele/oof_map3,0.87392
oof_acc,0.7925
oof_f1,0.78947
oof_map3,0.87392


Saved oof_ele.npy + test_ele.npy in /kaggle/working/artifacts/electra


NameError: name 'kagglehub' is not defined

In [29]:
import kagglehub

In [31]:
push_model(
    "electra",
    "/kaggle/working/artifacts/electra",
    "."
)

[hub] uploading 'electra' -> kamal134134/genai_project/pyTorch/electra
Uploading Model https://api.kaggle.com/models/kamal134134/genai_project/pyTorch/electra ...


Uploading: 100%|██████████| 10.1k/10.1k [00:00<00:00, 25.5kB/s]

Upload successful: /kaggle/working/upload_model/test_ele.npy (10KB)
Starting upload for file /kaggle/working/upload_model/best_electra_fold0.pth



Uploading: 100%|██████████| 1.34G/1.34G [00:12<00:00, 104MB/s] 

Upload successful: /kaggle/working/upload_model/best_electra_fold0.pth (1GB)
Starting upload for file /kaggle/working/upload_model/oof_ele.npy



Uploading: 100%|██████████| 40.1k/40.1k [00:00<00:00, 94.0kB/s]

Upload successful: /kaggle/working/upload_model/oof_ele.npy (39KB)
Starting upload for file /kaggle/working/upload_model/best_electra_fold1.pth



Uploading: 100%|██████████| 1.34G/1.34G [00:13<00:00, 100MB/s] 

Upload successful: /kaggle/working/upload_model/best_electra_fold1.pth (1GB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/kamal134134/genai_project/pyTorch/electra
Done. Inference notebooks can now kagglehub.


## DOWNLOADING MODEL FROM KAGGLEHUB

In [ ]:
import kagglehub

path = kagglehub.model_download("kamal134134/genai_project/pyTorch/electra")

print("Path to model files:", path)

## ELECTRA SUBMISSION

In [3]:
OPTS = ['A', 'B', 'C', 'D', 'E']

test_deb = np.load('/kaggle/input/models/kamal134134/genai_project/pytorch/electra/1/test_ele.npy')

order = (-test_deb).argsort(axis=1)
preds = [' '.join(OPTS[k] for k in order[i, :3]) for i in range(len(test))]

sub = pd.DataFrame({'ID': test['id'].values, 'Prediction': preds})
sub.to_csv('submission_electra.csv', index=False)
print("✓ Submission file for Deberta: submission_electra.csv")

✓ Submission file for Deberta: submission_electra.csv
